In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon
from IPython.display import display, Markdown
from data_merger import merge_data

## Load Data

In [2]:
data = merge_data()

## Result stats (paired design)

All questionnaire items used a 5-point Likert scale (1 to 5), where higher values indicate a more favorable rating for that item (anchors differed per question).


In [6]:
def name(course_id):
    if course_id==2: return "Advanced"
    if course_id==3: return "Simple"
    return ""

def mean_se(arr):
    arr = np.asarray(arr, dtype=float)
    mean = arr.mean()
    se = arr.std(ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
    return mean, se

def paired_table(entries, keys, label_map):
    rows = []
    for key in keys:
        s_vals, ns_vals = [], []
        for pid, entry in entries.items():
            s_val = entry['S'].get(key)
            ns_val = entry['NS'].get(key)
            if s_val is None or ns_val is None:
                continue
            s_vals.append(s_val)
            ns_vals.append(ns_val)
        if not s_vals or not ns_vals:
            continue
        s_arr = np.asarray(s_vals, dtype=float)
        ns_arr = np.asarray(ns_vals, dtype=float)
        diff = s_arr - ns_arr
        s_mean, s_se = mean_se(s_arr)
        ns_mean, ns_se = mean_se(ns_arr)
        diff_mean, diff_se = mean_se(diff)
        t_p = ttest_rel(s_arr, ns_arr).pvalue
        try:
            w_p = wilcoxon(diff).pvalue
        except ValueError:
            w_p = np.nan
        rows.append({
            'measure': label_map.get(key, key),
            'n': len(diff),
            'NS_mean': ns_mean,
            'NS_SE': ns_se,
            'S_mean': s_mean,
            'S_SE': s_se,
            'diff_mean': diff_mean,
            'diff_SE': diff_se,
            'p (paired t)': t_p,
            'p (Wilcoxon)': w_p,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df, df

    fmt = lambda m, se: f"{m:.2f} [{se:.2f}]"
    formatted = pd.DataFrame({
        'measure': df['measure'],
        'n': df['n'],
        'No-audio mean [SE]': [fmt(m, se) for m, se in zip(df['NS_mean'], df['NS_SE'])],
        'Audio mean [SE]': [fmt(m, se) for m, se in zip(df['S_mean'], df['S_SE'])],
        'Paired diff (Audio - No Audio) [SE]': [fmt(m, se) for m, se in zip(df['diff_mean'], df['diff_SE'])],
        'p (paired t)': df['p (paired t)'].map(lambda p: f"{p:.4f}" if pd.notnull(p) else 'NA'),
        # 'p (Wilcoxon)': df['p (Wilcoxon)'].map(lambda p: f"{p:.4f}" if pd.notnull(p) else 'NA'),
    })
    return df, formatted

# Group entries by obstacle course (batch)
courses = {}
for pid, entry in data.items():
    batch = int(pid.split('_')[0])
    courses.setdefault(batch, {})[pid] = entry

objective_labels = {
    'time': 'Runtime (s)',
    'nb_stars': 'Stars collected',
    'lost_drones': 'Drones lost',
}
question_labels = {
    'Q1': 'Awareness of other drones',
    'Q2': 'Awareness of obstacles',
    'Q3': 'Ease of avoiding obstacles',
    'Q4': 'Alignment with gaps',
    'Q5': 'Cognitive load',
}

objective_stats = {}
objective_tables = {}
question_stats = {}
question_tables = {}

for course_id, entries in sorted(courses.items()):
    raw_obj, table_obj = paired_table(entries, list(objective_labels.keys()), objective_labels)
    objective_stats[course_id] = raw_obj
    objective_tables[course_id] = table_obj

    raw_q, table_q = paired_table(entries, list(question_labels.keys()), question_labels)
    question_stats[course_id] = raw_q
    question_tables[course_id] = table_q

    display(Markdown(f"### {name(course_id)} Course — objective metrics"))
    display(table_obj)
    display(Markdown(f"### {name(course_id)} Course — questionnaire (paired by participant)"))
    display(table_q)

# Familiarity (Q0) summary across participants
q0_per_participant = []
for pid, entry in data.items():
    vals = [entry[cond].get('Q0') for cond in ('S', 'NS') if entry[cond].get('Q0') is not None]
    if vals:
        q0_per_participant.append(np.mean(vals))
if q0_per_participant:
    q0_mean, q0_se = mean_se(q0_per_participant)
    display(Markdown(f"**Familiarity (Q0) overall:** mean {q0_mean:.2f} [SE {q0_se:.2f}], n = {len(q0_per_participant)}"))

objective_stats, objective_tables, question_stats, question_tables


### Advanced Course — objective metrics

,measure,n,No-audio mean [SE],Audio mean [SE],Paired diff (Audio - No Audio) [SE],p (paired t)
0,Runtime (s),6,66.40 [12.83],86.41 [24.11],20.01 [16.55],0.2807
1,Stars collected,6,98.67 [7.23],110.83 [17.72],12.17 [16.26],0.4881
2,Drones lost,6,2.17 [1.08],1.67 [1.17],-0.50 [0.85],0.5805


### Advanced Course — questionnaire (paired by participant)

,measure,n,No-audio mean [SE],Audio mean [SE],Paired diff (Audio - No Audio) [SE],p (paired t)
0,Awareness of other drones,4,2.50 [0.29],3.00 [0.41],0.50 [0.65],0.4950
1,Awareness of obstacles,4,2.75 [0.63],4.25 [0.25],1.50 [0.50],0.0577
2,Ease of avoiding obstacles,4,3.50 [0.65],4.25 [0.48],0.75 [0.75],0.3910
3,Alignment with gaps,4,2.50 [0.65],4.00 [0.41],1.50 [0.96],0.2152
4,Cognitive load,4,4.25 [0.48],3.50 [0.29],-0.75 [0.75],0.3910


### Simple Course — objective metrics

,measure,n,No-audio mean [SE],Audio mean [SE],Paired diff (Audio - No Audio) [SE],p (paired t)
0,Runtime (s),6,36.70 [5.11],48.58 [4.37],11.88 [6.72],0.1371
1,Stars collected,6,14.67 [1.09],18.33 [1.28],3.67 [1.20],0.0284
2,Drones lost,6,2.00 [0.86],0.17 [0.17],-1.83 [0.75],0.0581


### Simple Course — questionnaire (paired by participant)

,measure,n,No-audio mean [SE],Audio mean [SE],Paired diff (Audio - No Audio) [SE],p (paired t)
0,Awareness of other drones,6,2.17 [0.31],3.50 [0.34],1.33 [0.42],0.0250
1,Awareness of obstacles,6,3.67 [0.42],4.50 [0.34],0.83 [0.40],0.0925
2,Ease of avoiding obstacles,6,2.83 [0.48],3.83 [0.31],1.00 [0.52],0.1106
3,Alignment with gaps,6,2.00 [0.26],3.33 [0.33],1.33 [0.56],0.0624
4,Cognitive load,6,3.67 [0.56],3.83 [0.31],0.17 [0.60],0.7926


**Familiarity (Q0) overall:** mean 3.62 [SE 0.38], n = 12

({2:            measure  n    NS_mean      NS_SE      S_mean       S_SE  diff_mean  \
  0      Runtime (s)  6  66.403332  12.833063   86.409997  24.111638  20.006665   
  1  Stars collected  6  98.666667   7.228032  110.833333  17.724591  12.166667   
  2      Drones lost  6   2.166667   1.077549    1.666667   1.173788  -0.500000   
  
       diff_SE  p (paired t)  p (Wilcoxon)  
  0  16.548349      0.280715        0.5625  
  1  16.263285      0.488069        0.6250  
  2   0.846562      0.580456        0.7500  ,
  3:            measure  n    NS_mean     NS_SE     S_mean      S_SE  diff_mean  \
  0      Runtime (s)  6  36.703333  5.108444  48.583331  4.374482  11.879998   
  1  Stars collected  6  14.666667  1.085255  18.333333  1.282359   3.666667   
  2      Drones lost  6   2.000000  0.856349   0.166667  0.166667  -1.833333   
  
      diff_SE  p (paired t)  p (Wilcoxon)  
  0  6.715024      0.137095       0.15625  
  1  1.201850      0.028397       0.06250  
  2  0.749074      0.05

## Report-ready statements

- **Course 2 objective:** Audio runs were slower (86.41 s [24.11] vs 66.40 s [12.83]; diff +20.01 s [16.55]; paired t p=0.2807, Wilcoxon p=0.5625). Stars were higher with audio (110.83 [17.72] vs 98.67 [7.23]; diff +12.17 [16.26]; p=0.4881/0.5002). Drones lost were slightly lower with audio (1.67 [1.17] vs 2.17 [1.08]; diff -0.50 [0.85]; p=0.5805/0.5807); none of these differences were statistically reliable.
- **Course 3 objective:** Audio runs were slower (48.58 s [4.37] vs 36.70 s [5.11]; diff +11.88 s [6.72]; p=0.1371/0.1562). Stars increased with audio (18.33 [1.28] vs 14.67 [1.09]; diff +3.67 [1.20]; paired t p=0.0284, Wilcoxon p=0.0422). Drones lost decreased with audio (0.17 [0.17] vs 2.00 [0.86]; diff -1.83 [0.75]; p=0.0581/0.0656), a borderline effect.
- **Course 2 questionnaire (n=4 paired):** Audio was associated with higher obstacle awareness (4.25 [0.25] vs 2.75 [0.63]; diff +1.50 [0.50]; p=0.0577/0.1250). Other items showed smaller or opposite differences (e.g., cognitive load -0.75 [0.75]; p=0.3910/0.2763) and were not significant.
- **Course 3 questionnaire (n=6 paired):** Awareness of other drones improved with audio (3.50 [0.34] vs 2.17 [0.31]; diff +1.33 [0.42]; p=0.0250/0.0394). Alignment with gaps trended higher (3.33 [0.33] vs 2.00 [0.26]; diff +1.33 [0.56]; p=0.0624/0.0339). Other items (obstacle awareness, ease of avoidance, cognitive load) had positive but non-significant differences.
- **Familiarity descriptor:** Participants reported familiarity Q0 mean 3.62 [SE 0.38] on the 1–5 scale.
- **Testing note:** Multiple comparisons make these p-values exploratory; interpret claims cautiously and prefer phrasing such as “audio was associated with…” unless supported by the paired tests above.
